# 04. Logistic Regression Modeling: Demand Surge Classification
## Retail Sales Predictive Analytics & Time Series Forecasting

*(Note: `notebooks/04_logistic_regression_modeling.ipynb` is the canonical Phase 2 modeling notebook, matching the repository's `03_linear_regression_modeling.ipynb` convention. `notebooks/04_logistic_regression_classification.ipynb` is maintained as a synchronized mirror.)*

### Phase 2 Objectives & Methodology:
1. **Problem Formulation:** Frame demand surge prediction as a binary classification task to mitigate retail stockout risk.
   - `1` = **Demand Surge**: Unit sales exceed the store-family historical 80th percentile threshold ($\text{sales} > \tau_{s,f}$).
   - `0` = **Normal Demand**: Unit sales are within normal historical operating bounds ($\text{sales} \le \tau_{s,f}$).
2. **Strict Leakage Prevention Protocol:**
   - All surge thresholds $\tau_{s,f}$ are derived **exclusively from historical training data** (`2016-08-01` to `2017-07-31`).
   - Sparse/inactive series with $\tau_{s,f} \le 0$ fall back to the product family training threshold, with a physical floor of $1.0$ unit.
   - Same-day sales and simultaneous transactions are strictly excluded from predictive features.
   - All lag features look back $\ge 16$ days matching the forecast horizon origin.
3. **Chronological Splitting (Option B: 1-Year Rolling Window):**
   - Training period: `2016-08-01` to `2017-07-31` (648,648 rows across 54 stores and 33 product families).
   - Validation period: `2017-08-01` to `2017-08-15` (26,730 rows, identical for all models).
4. **Out-of-Sample Threshold Calibration:**
   - The decision threshold is tuned on a held-out temporal partition within training data (June-July 2017: 108,702 rows) using a calibration model trained strictly on earlier observations (Aug 2016-May 2017: 539,946 rows). The calibration model has never seen the calibration rows during training.
5. **Algorithm Benchmarking:**
   - `DummyClassifier (Most Frequent)`: Zero-intelligence majority class baseline.
   - `Logistic Regression (Default Class Weighting, Threshold = 0.50)`: Unweighted standard classification.
   - `Logistic Regression (Balanced Class Weighting, Threshold = 0.50)`: Inverse class frequency weighting for stockout prevention.
   - `Logistic Regression (Tuned Decision Threshold, Threshold = 0.29)`: Out-of-sample calibrated decision threshold maximizing F1-score.
6. **Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, ROC-AUC, PR-AUC, and Confusion Matrix (TN, FP, FN, TP).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TARGET_COL, SURGE_TARGET_COL, MODELS_DIR, REPORTS_DIR, FIGURES_DIR
from src.data_loader import load_raw_datasets
from src.preprocess import (
    merge_external_features,
    add_lag_and_rolling_features,
    compute_store_family_baselines,
    compute_historical_surge_thresholds,
    create_surge_classification_target,
)
from src.logistic_regression import (
    LOGISTIC_NUMERIC_FEATURES,
    LOGISTIC_CATEGORICAL_FEATURES,
    build_logistic_regression_pipeline,
    find_optimal_threshold_on_train,
    train_and_evaluate_classifier,
    plot_confusion_matrices,
    plot_roc_and_pr_curves,
    plot_logistic_coefficients,
)

print("Environment initialized successfully.")

## 1. Data Ingestion & Leakage-Safe Feature Engineering
We load all primary Kaggle tables (`train.csv`, `stores.csv`, `oil.csv`, `holidays_events.csv`), clean macroeconomic signals, and extract calendar/payday indicators.

In [ ]:
print("Loading raw datasets...")
raw = load_raw_datasets()

print("Merging store metadata, oil prices, and calendar events...")
merged = merge_external_features(raw["train"], raw["stores"], raw["oil"], raw["holidays"])

print("Generating lag features (>= 16 days) and rolling averages...")
featured = add_lag_and_rolling_features(merged)
print(f"Total feature matrix shape: {featured.shape}")

## 2. Chronological Split & Leakage-Free Target Definition
We partition into a 1-year training set (`2016-08-01` to `2017-07-31`) and a 15-day holdout validation set (`2017-08-01` to `2017-08-15`).
Surge thresholds are derived **strictly from training data**.

In [ ]:
train_df = featured[(featured["date"] >= "2016-08-01") & (featured["date"] <= "2017-07-31")].copy()
val_df = featured[(featured["date"] >= "2017-08-01") & (featured["date"] <= "2017-08-15")].copy()

# Compute strictly past-only dynamic baselines
train_df, val_df = compute_store_family_baselines(train_df, val_df, full_df=featured, target_col=TARGET_COL)

# Derive surge thresholds exclusively from train_df
surge_threshold_map = compute_historical_surge_thresholds(train_df, quantile=0.80, target_col=TARGET_COL)
train_df = create_surge_classification_target(train_df, threshold_map=surge_threshold_map, target_col=TARGET_COL, surge_col=SURGE_TARGET_COL)
val_df = create_surge_classification_target(val_df, threshold_map=surge_threshold_map, target_col=TARGET_COL, surge_col=SURGE_TARGET_COL)

train_pos = train_df[SURGE_TARGET_COL].sum()
val_pos = val_df[SURGE_TARGET_COL].sum()

print(f"Training Rows:   {len(train_df):,} | Normal: {len(train_df)-train_pos:,} ({(1-train_pos/len(train_df)):.2%}) | Surge: {train_pos:,} ({train_pos/len(train_df):.2%})")
print(f"Validation Rows: {len(val_df):,} | Normal: {len(val_df)-val_pos:,} ({(1-val_pos/len(val_df)):.2%}) | Surge: {val_pos:,} ({val_pos/len(val_df):.2%})")

feature_cols = LOGISTIC_NUMERIC_FEATURES + LOGISTIC_CATEGORICAL_FEATURES
X_train = train_df[feature_cols].copy()
y_train = train_df[SURGE_TARGET_COL].values
X_val = val_df[feature_cols].copy()
y_val = val_df[SURGE_TARGET_COL].values

for col in LOGISTIC_CATEGORICAL_FEATURES:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

## 3. Algorithm Benchmarking & Training
We train DummyClassifier, Logistic Regression (Default), and Logistic Regression (Balanced) on the training set, and derive an optimal decision threshold using strictly out-of-sample calibration within training data.

In [ ]:
model_definitions = [
    {"name": "DummyClassifier (Most Frequent)", "model_type": "dummy", "params": {}, "threshold": 0.5},
    {"name": "Logistic Regression (Default, Threshold = 0.5)", "model_type": "default", "params": {"class_weight": None}, "threshold": 0.5},
    {"name": "Logistic Regression (Balanced, Threshold = 0.5)", "model_type": "balanced", "params": {"class_weight": "balanced"}, "threshold": 0.5},
]

benchmark_results = []
cm_dict = {}
plot_results = {}
pipelines = {}

for m_def in model_definitions:
    name = m_def["name"]
    print(f"Training {name}...")
    pipe = build_logistic_regression_pipeline(
        numeric_features=LOGISTIC_NUMERIC_FEATURES,
        categorical_features=LOGISTIC_CATEGORICAL_FEATURES,
        model_type=m_def["model_type"],
        **m_def["params"],
    )
    
    metrics, y_proba, y_pred = train_and_evaluate_classifier(
        pipeline=pipe,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        threshold=m_def["threshold"],
    )
    
    benchmark_results.append({"Model": name, **metrics})
    pipelines[name] = pipe
    cm_dict[name] = np.array([
        [metrics["True_Negatives"], metrics["False_Positives"]],
        [metrics["False_Negatives"], metrics["True_Positives"]],
    ])
    plot_results[name] = {"metrics": metrics, "proba": y_proba, "pred": y_pred}

# Optimize threshold strictly using out-of-sample calibration within training data
default_pipe = pipelines["Logistic Regression (Default, Threshold = 0.5)"]
opt_thresh, train_sub_f1 = find_optimal_threshold_on_train(
    X_train=X_train,
    y_train=y_train,
    train_dates=train_df["date"],
    cutoff_date="2017-06-01",
    numeric_features=LOGISTIC_NUMERIC_FEATURES,
    categorical_features=LOGISTIC_CATEGORICAL_FEATURES,
    model_type="default",
)

# Evaluate with out-of-sample calibrated threshold on validation data
y_val_proba = plot_results["Logistic Regression (Default, Threshold = 0.5)"]["proba"]
y_val_pred_opt = (y_val_proba >= opt_thresh).astype(int)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
cm_opt = confusion_matrix(y_val, y_val_pred_opt)

tuned_name = f"Logistic Regression (Tuned Threshold = {opt_thresh:.2f})"
tuned_metrics = {
    "Model": tuned_name,
    "Accuracy": round(float(accuracy_score(y_val, y_val_pred_opt)), 4),
    "Precision": round(float(precision_score(y_val, y_val_pred_opt, zero_division=0)), 4),
    "Recall": round(float(recall_score(y_val, y_val_pred_opt, zero_division=0)), 4),
    "F1": round(float(f1_score(y_val, y_val_pred_opt, zero_division=0)), 4),
    "ROC_AUC": round(float(roc_auc_score(y_val, y_val_proba)), 4),
    "PR_AUC": round(float(average_precision_score(y_val, y_val_proba)), 4),
    "Threshold": round(opt_thresh, 4),
    "True_Negatives": int(cm_opt[0, 0]),
    "False_Positives": int(cm_opt[0, 1]),
    "False_Negatives": int(cm_opt[1, 0]),
    "True_Positives": int(cm_opt[1, 1]),
    "Fit_Time_Sec": plot_results["Logistic Regression (Default, Threshold = 0.5)"]["metrics"]["Fit_Time_Sec"],
}
benchmark_results.append(tuned_metrics)
cm_dict[tuned_name] = cm_opt
plot_results[tuned_name] = {"metrics": tuned_metrics, "proba": y_val_proba, "pred": y_val_pred_opt}

results_df = pd.DataFrame(benchmark_results)
display_cols = ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "Threshold", "Fit_Time_Sec"]
print("\n--- Benchmark Comparison Table ---")
print(results_df[display_cols].to_string(index=False))

## 4. Diagnostic Visualizations: Confusion Matrices, ROC, and PR Curves

In [ ]:
# Plot side-by-side Confusion Matrices
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5), sharey=True)
for ax, (model_name, cm) in zip(axes, cm_dict.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Normal (0)", "Surge (1)"],
                yticklabels=["Normal (0)", "Surge (1)"],
                annot_kws={"size": 10, "weight": "bold"})
    ax.set_title(f"{model_name}", fontsize=10, fontweight="bold", pad=8)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()

# Plot ROC and PR Curves
from src.logistic_regression import plot_roc_and_pr_curves
roc_pr_path = FIGURES_DIR / "logistic_regression_roc_pr_curves.png"
plot_roc_and_pr_curves(plot_results, y_val, roc_pr_path)
from IPython.display import Image, display
display(Image(filename=str(roc_pr_path)))

## 5. Feature Importance: Logistic Regression Odds Ratios

In [ ]:
coef_path = FIGURES_DIR / "logistic_regression_coefficients.png"
plot_logistic_coefficients(default_pipe, LOGISTIC_NUMERIC_FEATURES, LOGISTIC_CATEGORICAL_FEATURES, coef_path, top_n=20)
display(Image(filename=str(coef_path)))

## 6. Academic Summary & Supply Chain Decision-Making

1. **Accuracy is Deceptive in Retail Event Forecasting:**
   - The DummyClassifier achieves 83.30% Accuracy but has 0.00% Recall, missing 100% of demand surges (4,463 stockouts).
2. **False Alarms vs. Missed Surges Trade-off:**
   - **Default Logistic Regression (Threshold = 0.50):** Highly conservative (Precision 45.24%, 1,817 false alarms), but misses 2,962 surges (Recall 33.63%).
   - **Balanced Logistic Regression (Threshold = 0.50):** Aggressive stockout prevention (Recall 76.94%, 3,434 surges captured), but incurs 9,259 false alarms (Precision 27.05%).
   - **Out-of-Sample Tuned Threshold (0.29):** Calibrated strictly out-of-sample on training data. Achieves the highest overall F1-score (0.4241) and captures 55.90% of surges with significantly fewer false alarms (4,808 FP).
3. **Zero Target Leakage & Out-of-Sample Calibration:**
   - Surge thresholds are derived strictly on historical training observations.
   - Calibration model was fit only on `2016-08-01` to `2017-05-31` and evaluated on `2017-06-01` to `2017-07-31` to select the threshold without using in-sample predictions.
   - Identical validation holdout rows are evaluated across all algorithms without data leakage.